> **Estado de validación:** Ejecutado: interfaces locales y pruebas reales de chat, streaming, memoria, menú y prompt personalizado.

# Construyendo aplicaciones de IA con Gradio y NVIDIA NIM

En los notebooks anteriores interactuamos con LLMs directamente desde Python.
En este notebook construiremos interfaces web interactivas usando **Gradio**,
conectadas a **NVIDIA NIM** y al modelo **Nemotron 3 Super**.

Gradio permite crear interfaces web para modelos de IA con muy poco código. Es
muy útil para demostraciones, prototipos y herramientas internas.

## Prerrequisitos

Asegúrate de tener `NVIDIA_API_KEY` en el archivo `.env`. Si todavía no la
tienes, revisa las instrucciones del notebook 03.

In [1]:
# Dependencias instaladas desde requirements.txt.

## 1. Configuración del Entorno

In [2]:
import os
import gradio as gr
from openai import OpenAI
from dotenv import load_dotenv

/Users/fabiolavalencia/Documents/Codex/2026-10-06/bue/work/venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
load_dotenv()
assert os.getenv("NVIDIA_API_KEY"), "Configura NVIDIA_API_KEY en .env"
print("NVIDIA API Key cargada correctamente")

NVIDIA API Key cargada correctamente


In [4]:
# Cliente compatible con OpenAI conectado a NVIDIA NIM
client = OpenAI(timeout=90, max_retries=1,
    base_url="https://integrate.api.nvidia.com/v1",
    api_key=os.getenv("NVIDIA_API_KEY"),
)

In [5]:
# Modelo usado en todo el notebook
MODELO = "nvidia/nemotron-3-super-120b-a12b"

## 2. Tu primera interfaz con Gradio

Antes de conectar el LLM, entendamos cómo funciona Gradio con un ejemplo simple.

`gr.Interface` toma una función de Python y la convierte en una interfaz web:

- `fn`: función que se ejecutará.
- `inputs`: componentes de entrada.
- `outputs`: componentes de salida.

In [6]:
# Función simple que devuelve lo que el usuario escribe
def echo(message):
    return f"Dijiste: {message}"

In [7]:
# Creamos la interfaz — Gradio levanta un servidor web local automáticamente
# Al ejecutar esta celda verás un link para abrir la interfaz en el navegador
demo = gr.Interface(
    fn=echo,
    inputs="textbox",
    outputs="textbox",
    title="Echo Bot",
    description="Una interfaz simple que repite lo que escribes",
    flagging_mode="never"  # desactiva el botón de reportar respuestas
)

In [8]:
demo.launch(
    server_name="127.0.0.1",
    server_port=8081,
    show_error=True, prevent_thread_lock=True, share=False
)

* Running on local URL:  http://127.0.0.1:8081
* To create a public link, set `share=True` in `launch()`.


In [9]:
demo.close()

Closing server running on port: 8081


## 3. Chatbot básico con NVIDIA NIM

Ahora conectamos Gradio con NVIDIA NIM. La función `get_response` llama a la API
y devuelve el texto; Gradio se encarga de presentarlo en la interfaz.

In [10]:
def get_response(message: str) -> str:
    """Obtiene una respuesta de NVIDIA NIM."""
    response = client.chat.completions.create(
        model=MODELO,
        max_tokens=2048,
        extra_body={"chat_template_kwargs": {"enable_thinking": False}},
        messages=[
            {"role": "system", "content": "Eres un asistente útil."},
            {"role": "user", "content": message},
        ],
        temperature=1.0,
        top_p=0.95,
    )
    return response.choices[0].message.content

In [11]:
chatbot = gr.Interface(
    fn=get_response,
    inputs=gr.Textbox(placeholder="Escribe tu pregunta aquí..."),
    outputs="text",
    title="Chatbot con NVIDIA NIM",
    description="Conversa con NVIDIA Nemotron 3 Super.",
    flagging_mode="never",
)

In [12]:
# La interfaz se lanza en la siguiente celda.

In [13]:
chatbot.launch(
    server_name="127.0.0.1",
    server_port=8081,
    show_error=True, prevent_thread_lock=True, share=False
)

* Running on local URL:  http://127.0.0.1:8081
* To create a public link, set `share=True` in `launch()`.


In [14]:
chatbot.close()

Closing server running on port: 8081


## 4. Chatbot con Streaming

El streaming muestra la respuesta a medida que se genera, token por token.
En Gradio esto se logra con `yield` en lugar de `return` — la función se convierte
en un generador que devuelve la respuesta parcial en cada paso.

In [15]:
def stream_response(message: str):
    """Genera una respuesta de NVIDIA NIM en modo streaming."""
    respuesta = ""

    stream = client.chat.completions.create(
        model=MODELO,
        max_tokens=2048,
        extra_body={"chat_template_kwargs": {"enable_thinking": False}},
        messages=[
            {"role": "system", "content": "Eres un asistente útil."},
            {"role": "user", "content": message},
        ],
        temperature=1.0,
        top_p=0.95,
        stream=True,
    )

    for chunk in stream:
        texto = chunk.choices[0].delta.content if chunk.choices else None
        if texto:
            respuesta += texto
            yield respuesta

In [16]:
# Gradio detecta automáticamente que la función es un generador (usa yield)
# y activa el modo streaming en la interfaz
streaming_chatbot = gr.Interface(
    fn=stream_response,
    inputs="textbox",
    outputs="text",
    title="Chatbot con Streaming",
    description="Las respuestas aparecen en tiempo real a medida que se generan",
    flagging_mode="never"
)

In [17]:
streaming_chatbot.launch(
    server_name="127.0.0.1",
    server_port=8081,
    show_error=True, prevent_thread_lock=True, share=False
)

* Running on local URL:  http://127.0.0.1:8081
* To create a public link, set `share=True` in `launch()`.


In [18]:
streaming_chatbot.close()

Closing server running on port: 8081


## 5. Chatbot con memoria

`gr.ChatInterface` está diseñado para chatbots con historial. Gradio gestiona la
interfaz y entrega a nuestra función el mensaje actual y los intercambios
anteriores. Nosotros convertimos ese historial al formato estándar de mensajes
de la API.

In [19]:
def get_text(content):
    """Extrae texto de los formatos de contenido usados por Gradio."""
    if isinstance(content, str):
        return content
    if isinstance(content, list) and content:
        first = content[0]
        return first.get("text", "") if isinstance(first, dict) else str(first)
    return str(content)


def history_to_messages(history: list) -> list:
    """Convierte historiales nuevos o antiguos de Gradio a mensajes OpenAI."""
    messages = []
    for entry in history:
        if isinstance(entry, dict):
            messages.append({
                "role": entry["role"],
                "content": get_text(entry["content"]),
            })
        elif isinstance(entry, (list, tuple)) and len(entry) == 2:
            user_msg, assistant_msg = entry
            messages.append({"role": "user", "content": get_text(user_msg)})
            if assistant_msg:
                messages.append({
                    "role": "assistant",
                    "content": get_text(assistant_msg),
                })
    return messages


def chat_con_memoria(message: str, history: list):
    messages = [{"role": "system", "content": "Eres un asistente útil."}]
    messages.extend(history_to_messages(history))
    messages.append({"role": "user", "content": message})

    respuesta = ""
    stream = client.chat.completions.create(
        model=MODELO,
        max_tokens=2048,
        extra_body={"chat_template_kwargs": {"enable_thinking": False}},
        messages=messages,
        temperature=1.0,
        top_p=0.95,
        stream=True,
    )
    for chunk in stream:
        texto = chunk.choices[0].delta.content if chunk.choices else None
        if texto:
            respuesta += texto
            yield respuesta

In [20]:
# gr.ChatInterface se encarga de toda la UI del chat:
# burbuja de mensajes, historial visual, campo de entrada, etc.
memory_chatbot = gr.ChatInterface(
    fn=chat_con_memoria,
    title="Chatbot con Memoria",
    description="El asistente recuerda el contexto de toda la conversacion.",
    examples=[
        "Cuentame sobre el aprendizaje automatico",
        "Como funcionan las redes neuronales?",
        "Cual es la diferencia entre IA y ML?"
    ],
    flagging_mode="never"
)

In [21]:
memory_chatbot.launch(
    server_name="127.0.0.1",
    server_port=8081,
    show_error=True, prevent_thread_lock=True, share=False
)

* Running on local URL:  http://127.0.0.1:8081
* To create a public link, set `share=True` in `launch()`.


In [22]:
memory_chatbot.close()

Closing server running on port: 8081


## 6. Generador de Menús de Restaurante

Veamos un ejemplo de aplicación con múltiples entradas. Gradio puede manejar
formularios complejos — aquí el usuario proporciona tres campos y obtiene
un menú generado por IA en formato Markdown.

In [23]:
def generate_menu(nombre_restaurante: str, tipo_cocina: str, requisitos_especiales: str = "Ninguno") -> str:
    """Genera un menú de restaurante usando NVIDIA NIM."""
    prompt = f"""
    Crea un menú para "{nombre_restaurante}", un restaurante de cocina {tipo_cocina}.
    Requisitos especiales: {requisitos_especiales}

    Incluye 3 entradas, 4 platos principales y 2 postres. Para cada elemento
    incluye nombre, descripción breve y precio en pesos colombianos. Formatea
    la respuesta en Markdown y agrega una introducción corta.
    """

    response = client.chat.completions.create(
        model=MODELO,
        max_tokens=2048,
        extra_body={"chat_template_kwargs": {"enable_thinking": False}},
        messages=[
            {
                "role": "system",
                "content": "Eres un consultor experto en restaurantes y diseño de menús.",
            },
            {"role": "user", "content": prompt},
        ],
        temperature=1.0,
        top_p=0.95,
    )
    return response.choices[0].message.content

In [24]:
# Interfaz con múltiples inputs y output en Markdown
menu_generator = gr.Interface(
    fn=generate_menu,
    inputs=[
        gr.Textbox(label="Nombre del Restaurante"),
        gr.Textbox(label="Tipo de Cocina (ej. italiana, japonesa, colombiana)"),
        gr.Textbox(
            label="Requisitos Especiales (Opcional)",
            placeholder="ej. opciones vegetarianas, sin gluten"
        )
    ],
    outputs=gr.Markdown(label="Menu Generado"),
    title="Generador de Menus con IA",
    description="Crea un menu profesional para tu restaurante en segundos",
    flagging_mode="never"
)

In [25]:
menu_generator.launch(
    server_name="127.0.0.1",
    server_port=8081,
    show_error=True, prevent_thread_lock=True, share=False
)

* Running on local URL:  http://127.0.0.1:8081
* To create a public link, set `share=True` in `launch()`.


In [26]:
menu_generator.close()

Closing server running on port: 8081


## 7. Personalizando el System Prompt

Una extensión útil: permitir al usuario definir el comportamiento del asistente
cambiando el system prompt desde la interfaz.

In [27]:
def chat_personalizable(message: str, history: list, system_prompt: str):
    """Chat en el que el usuario personaliza el system prompt."""
    system_prompt = system_prompt or "Eres un asistente útil."

    messages = [{"role": "system", "content": system_prompt}]
    messages.extend(history_to_messages(history))
    messages.append({"role": "user", "content": message})

    respuesta = ""
    stream = client.chat.completions.create(
        model=MODELO,
        max_tokens=2048,
        extra_body={"chat_template_kwargs": {"enable_thinking": False}},
        messages=messages,
        temperature=1.0,
        top_p=0.95,
        stream=True,
    )
    for chunk in stream:
        texto = chunk.choices[0].delta.content if chunk.choices else None
        if texto:
            respuesta += texto
            yield respuesta

In [28]:
# Usamos gr.ChatInterface con additional_inputs para agregar el campo de system prompt
custom_chatbot = gr.ChatInterface(
    fn=chat_personalizable,
    title="Chatbot Personalizable",
    description="Define como quieres que se comporte el asistente.",
    additional_inputs=[
        gr.Textbox(
            label="System Prompt",
            placeholder="ej. Eres un experto en Python que responde solo con codigo",
            value="Eres un asistente util y conciso."
        )
    ],
    examples=[
        ["Explica que es una red neuronal", "Eres un profesor universitario."],
        ["Escribe un haiku sobre la programacion", "Eres un poeta."],
    ],
    flagging_mode="never"
)

In [29]:
custom_chatbot.launch(
    server_name="127.0.0.1",
    server_port=8081,
    show_error=True, prevent_thread_lock=True, share=False
)

* Running on local URL:  http://127.0.0.1:8081
* To create a public link, set `share=True` in `launch()`.


In [30]:
custom_chatbot.close()

Closing server running on port: 8081


## Configuración de esta ejecución

Se carga la clave desde `.env` local, sin imprimirla. Se limita cada salida a 2048 tokens y se desactiva el modo de razonamiento para estas demostraciones breves. Las interfaces escuchan únicamente en 127.0.0.1 y se cierran tras cada ejemplo.

In [31]:
# Pruebas de las funciones reales que utilizan las interfaces.
print("Chat básico:", get_response("Responde en una frase: ¿qué es un modelo de IA?"))
streamed = list(stream_response("Escribe un saludo breve en español."))
assert streamed and streamed[-1].strip()
print("Streaming:", streamed[-1])
history = [{"role":"user","content":"Mi palabra de prueba es girasol."},
           {"role":"assistant","content":"Recordaré girasol."}]
remembered = list(chat_con_memoria("¿Cuál es mi palabra de prueba?",history))
assert remembered and "girasol" in remembered[-1].lower()
print("Memoria:", remembered[-1])
print("Menú:\n", generate_menu("Jardín de prueba","colombiana","Opciones vegetarianas"))
custom = list(chat_personalizable("Explica una red neuronal en una oración.", [],
    "Eres un profesor de IA. Responde brevemente en español."))
assert custom and custom[-1].strip()
print("Prompt personalizado:", custom[-1])


Chat básico: Un modelo de IA es un sistema matemático entrenado con datos para reconocer patrones y tomar decisiones o generar salidas sin intervención humana explícita.


Streaming: ¡Hola! 😊


Memoria: Tu palabra de prueba es girasol.


Menú:
 # Jardín de prueba  
*Donde la tierra colombiana se convierte en sabor*

En el corazón de una oasis verde, **Jardín de prueba** rinde homenaje a la riqueza gastronómica de Colombia, reinterpretando sus tradiciones con un enfoque consciente y vegetariano. Cada plato está pensado para celebrar los ingredientes nativos — tubérculos, frutas tropicales, legumbres y hierbas aromáticas — sin carne, pero con toda la profundidad, el alma y el color de nuestra tierra. Ven a descubrir cómo lo vegetal puede ser igualmente abundante, reconfortante y festivo.

---

## Entradas

**1. Arepa de choclo con queso campesino y guacamole de aguacate criollo**  
Suave arepa hecha de choclo molido, horneada hasta quedar dorada por fuera y esponjosa por dentro, rellena de queso campesino fundido y acompañada de un guacamole fresco con cebolla morada, cilantro y un toque de limón.  
*Precio: 18,000 COP*

**2. Empanadas de lentejas y plátano verde al estilo pacífico**  
Masita crujiente de plátano verde r

Prompt personalizado: Una red neuronal es un sistema computacional inspirado en el cerebro humano que aprende patrones de datos mediante capas interconectadas de nodos que ajustan sus conexiones a través de entrenamiento.
